# traffic_sign_detection_resnet_optimized_v2.ipynb

This notebook includes:
- KerasTuner hyperparameter search (ResNet50 baseline)
- Fine-tuning & partial unfreeze for ResNet50, EfficientNetB0, MobileNetV3Small
- 3-model ensemble + weighted ensemble optimization
- Test-Time Augmentation (TTA)
- Confusion matrix & per-class metrics
- Grad-CAM++ visualizations
- Final inference cell for single image / folder with Grad-CAM overlay

**Notes**: Update `DATA_DIR` to point to your dataset folder.

In [ ]:

import os, shutil, random, math, json, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV3Small, resnet50, efficientnet, mobilenet_v3
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix
print('TF', tf.__version__)


In [ ]:

DATA_DIR = r"D:\CGC\PHD\conf\paper\traffic_sign_detection_gtsrb\data"
TRAIN_DIR = os.path.join(DATA_DIR, "Train")
TEST_DIR = os.path.join(DATA_DIR, "Test")

IMG_SIZE = (96,96)
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42
os.makedirs('models', exist_ok=True)
NUM_CLASSES = len(next(os.walk(TRAIN_DIR))[1])
print("NUM_CLASSES =", NUM_CLASSES)


In [ ]:

# Create train_split and val_split (80/20)
def create_split(src_dir, out_train, out_val, val_ratio=0.2, seed=SEED):
    if os.path.exists(out_train) and os.path.exists(out_val):
        print('Split exists, skipping.')
        return
    os.makedirs(out_train, exist_ok=True)
    os.makedirs(out_val, exist_ok=True)
    classes = [d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    for cls in classes:
        files = [os.path.join(src_dir, cls, f) for f in os.listdir(os.path.join(src_dir, cls)) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        files.sort()
        random.Random(seed).shuffle(files)
        n_val = int(len(files)*val_ratio)
        val = files[:n_val]
        train = files[n_val:]
        os.makedirs(os.path.join(out_train, cls), exist_ok=True)
        os.makedirs(os.path.join(out_val, cls), exist_ok=True)
        for f in train:
            shutil.copy(f, os.path.join(out_train, cls, os.path.basename(f)))
        for f in val:
            shutil.copy(f, os.path.join(out_val, cls, os.path.basename(f)))
    print('Created split.')

create_split(TRAIN_DIR, os.path.join(DATA_DIR,'train_split'), os.path.join(DATA_DIR,'val_split'), val_ratio=0.2)


In [ ]:

# Diagnostics: class distribution & sample images
train_split = os.path.join(DATA_DIR,'train_split')
val_split = os.path.join(DATA_DIR,'val_split')
def counts(folder):
    classes = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder,d))]
    return {c: len([f for f in os.listdir(os.path.join(folder,c)) if f.lower().endswith(('.png','.jpg'))]) for c in classes}

tc = counts(train_split)
vc = counts(val_split)
df = pd.DataFrame({'class':list(tc.keys()), 'train':list(tc.values()), 'val':[vc.get(k,0) for k in tc.keys()]}).sort_values('train',ascending=False)
display(df.head(10))

plt.figure(figsize=(14,4)); plt.bar(df['class'].astype(str), df['train']); plt.xticks(rotation=90); plt.title('Train samples per class'); plt.show()

# show samples for first 9 classes
import matplotlib.image as mpimg
sample_classes = list(tc.keys())[:9]
plt.figure(figsize=(10,8))
for i, c in enumerate(sample_classes):
    imgs = [f for f in os.listdir(os.path.join(train_split,c)) if f.lower().endswith(('.png','.jpg'))]
    if not imgs: continue
    img = mpimg.imread(os.path.join(train_split,c,imgs[0]))
    plt.subplot(3,3,i+1); plt.imshow(img); plt.title(f'class {c} ({len(imgs)})'); plt.axis('off')
plt.show()


In [ ]:

train_datagen = ImageDataGenerator(preprocessing_function=resnet50.preprocess_input,
                                   rotation_range=15, width_shift_range=0.12, height_shift_range=0.12,
                                   shear_range=0.12, zoom_range=0.12, brightness_range=(0.7,1.3),
                                   horizontal_flip=False, fill_mode='reflect')

val_datagen = ImageDataGenerator(preprocessing_function=resnet50.preprocess_input)

train_gen = train_datagen.flow_from_directory(os.path.join(DATA_DIR,'train_split'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True, seed=SEED)
val_gen = val_datagen.flow_from_directory(os.path.join(DATA_DIR,'val_split'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False, seed=SEED)


In [ ]:

def build_model(base_name='resnet', input_shape=(96,96,3), num_classes=NUM_CLASSES, dropout=0.4):
    inp = Input(shape=input_shape)
    if base_name=='resnet':
        base = ResNet50(include_top=False, weights='imagenet', input_tensor=inp)
    elif base_name=='eff':
        base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inp)
    elif base_name=='mobile':
        base = MobileNetV3Small(include_top=False, weights='imagenet', input_tensor=inp)
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(dropout)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    return model, base


In [ ]:

# KerasTuner: tune learning_rate, dropout, dense_units (on ResNet baseline)
!pip install -q -U keras-tuner
import kerastuner as kt

def build_tunable(hp):
    lr = hp.Float('lr', 1e-5, 1e-3, sampling='log')
    dropout = hp.Float('dropout', 0.2, 0.5, step=0.05)
    dense_units = hp.Choice('dense_units', [64,128,256])
    model, base = build_model('resnet', dropout=dropout)
    for layer in base.layers:
        layer.trainable = False
    x = model.output
    # replace last Dense with hp choice
    x = Dense(dense_units, activation='relu')(x)
    x = Dropout(dropout)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=model.inputs, outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05), metrics=['accuracy'])
    return model

tuner = kt.RandomSearch(build_tunable, objective='val_accuracy', max_trials=12, executions_per_trial=1, directory='kt_dir', project_name='resnet_tune')
tuner.search(train_gen, validation_data=val_gen, epochs=8)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print('Best HPs:', best_hps.values)
# Save best hps
with open('models/best_hps.json','w') as f:
    json.dump(best_hps.values, f)


In [ ]:

# Train ResNet, EfficientNetB0, MobileNetV3Small with partial unfreeze as decided

def train_and_finetune(name, base_unfreeze_last=60, initial_epochs=6, fine_tune_epochs=24, batch_size=BATCH_SIZE, dropout=0.4, lr_head=1e-3, lr_ft=5e-5):
    print('\n== Training', name, '==')
    model, base = build_model('resnet' if name=='resnet' else ('eff' if name=='eff' else 'mobile'), dropout=dropout)
    for layer in base.layers:
        layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=lr_head), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05), metrics=['accuracy'])
    cb = [ModelCheckpoint(f'models/{name}_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
          EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
          CSVLogger(f'models/{name}_log.csv')]
    model.fit(train_gen, validation_data=val_gen, epochs=initial_epochs, callbacks=cb)
    # fine-tune
    # unfreeze last N layers of base
    for layer in base.layers[:-base_unfreeze_last]:
        layer.trainable = False
    for layer in base.layers[-base_unfreeze_last:]:
        layer.trainable = True
    model.compile(optimizer=Adam(learning_rate=lr_ft), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05), metrics=['accuracy'])
    model.fit(train_gen, validation_data=val_gen, epochs=fine_tune_epochs, callbacks=cb)
    model.save(f'models/{name}_finetuned.h5')
    print('Saved', f'models/{name}_finetuned.h5')

# Train with chosen strategies
train_and_finetune('resnet', base_unfreeze_last=60, initial_epochs=6, fine_tune_epochs=24)
train_and_finetune('eff', base_unfreeze_last=50, initial_epochs=6, fine_tune_epochs=24)
train_and_finetune('mobile', base_unfreeze_last=40, initial_epochs=6, fine_tune_epochs=24)


In [ ]:

# Ensemble weight sweep on validation set to find best weights
import itertools
models_files = {'resnet':'models/resnet_finetuned.h5','eff':'models/eff_finetuned.h5','mobile':'models/mobile_finetuned.h5'}
loaded = {}
for k,v in models_files.items():
    if os.path.exists(v):
        loaded[k] = load_model(v)
    else:
        print('Missing', v)

# create validation file list and labels
val_files = []
val_labels = []
class_indices = val_gen.class_indices
inv_map = {v:k for k,v in class_indices.items()}
for i in range(len(val_gen.filenames)):
    fname = val_gen.filepaths[i]
    val_files.append(fname)
    val_labels.append(np.argmax(val_gen.labels[i]))

def predict_model(model, preprocess_fn, files):
    preds = []
    for f in files:
        import cv2
        img = cv2.imread(f); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB); img = cv2.resize(img, IMG_SIZE)
        x = np.expand_dims(img,0).astype('float32')
        if preprocess_fn=='resnet': x = resnet50.preprocess_input(x)
        elif preprocess_fn=='eff': x = efficientnet.preprocess_input(x)
        else: x = mobilenet_v3.preprocess_input(x)
        preds.append(model.predict(x, verbose=0)[0])
    return np.vstack(preds)

preds = {}
if 'resnet' in loaded: preds['resnet'] = predict_model(loaded['resnet'], 'resnet', val_files)
if 'eff' in loaded: preds['eff'] = predict_model(loaded['eff'], 'eff', val_files)
if 'mobile' in loaded: preds['mobile'] = predict_model(loaded['mobile'], 'mobile', val_files)

best_acc = 0; best_w = None
weights_range = np.linspace(0,1,21)
for w1 in weights_range:
    for w2 in weights_range:
        for w3 in weights_range:
            ws = np.array([w1,w2,w3])
            if ws.sum()==0: continue
            ws = ws/ws.sum()
            p = 0
            if 'resnet' in preds: p = p + ws[0]*preds.get('resnet',0)
            if 'eff' in preds: p = p + ws[1]*preds.get('eff',0)
            if 'mobile' in preds: p = p + ws[2]*preds.get('mobile',0)
            y_pred = np.argmax(p, axis=1)
            acc = np.mean(y_pred == np.array(val_labels))
            if acc>best_acc:
                best_acc = acc; best_w = ws.copy()
best_acc, best_w


In [ ]:

# Confusion matrix and per-class metrics for the ensemble (using best weights found above)
if 'best_w' in globals() and 'preds' in globals():
    p = 0
    if 'resnet' in preds: p = p + best_w[0]*preds.get('resnet',0)
    if 'eff' in preds: p = p + best_w[1]*preds.get('eff',0)
    if 'mobile' in preds: p = p + best_w[2]*preds.get('mobile',0)
    y_pred = np.argmax(p, axis=1)
    y_true = np.array(val_labels)
    print('Ensemble accuracy on val =', np.mean(y_pred==y_true))
    print(classification_report(y_true, y_pred, digits=4))
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    import seaborn as sns
    plt.figure(figsize=(10,8)); sns.heatmap(cm, cmap='viridis'); plt.title('Normalized Confusion Matrix (ensemble)'); plt.show()
else:
    print('Run ensemble sweep first to compute best_w and preds.')


In [ ]:

# Grad-CAM++ style multi-layer visualization for a given image and models
import cv2
def gradcam_layers(model, img, layer_names):
    activations = []
    for lname in layer_names:
        grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(lname).output, model.output])
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img)
            pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]
        grads = tape.gradient(class_channel, conv_outputs)
        pooled = tf.reduce_mean(grads, axis=(0,1,2))
        conv = conv_outputs[0]
        heatmap = conv @ pooled[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = tf.maximum(heatmap,0)/(tf.math.reduce_max(heatmap)+1e-8)
        activations.append(heatmap.numpy())
    # average activations resized to image
    avg = np.mean([cv2.resize(h, IMG_SIZE) for h in activations], axis=0)
    return avg

# Example: pick a sample file and show Grad-CAM from each model
sample = None
val_files_list = sorted(glob.glob(os.path.join(TEST_DIR,'*.png')))
if len(val_files_list)>0:
    sample = val_files_list[0]
if sample and os.path.exists('models/resnet_finetuned.h5'):
    img = cv2.imread(sample); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB); img_r = cv2.resize(img, IMG_SIZE)
    x = np.expand_dims(img_r,0).astype('float32'); x_r = resnet50.preprocess_input(x.copy())
    mres = load_model('models/resnet_finetuned.h5')
    layers = ['conv3_block4_out','conv4_block6_out','conv5_block3_out'] if 'conv3_block4_out' in [l.name for l in mres.layers] else [l.name for l in mres.layers if 'conv' in l.name][-3:]
    heat = gradcam_layers(mres, x_r, layers)
    heat = cv2.resize(heat, (img_r.shape[1], img_r.shape[0]))
    heat_img = np.uint8(255*heat)
    heat_img = cv2.applyColorMap(heat_img, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_r, 0.6, heat_img, 0.4, 0)
    plt.figure(figsize=(8,4)); plt.subplot(1,2,1); plt.imshow(img_r); plt.axis('off'); plt.title('Image')
    plt.subplot(1,2,2); plt.imshow(overlay); plt.axis('off'); plt.title('Grad-CAM++ avg'); plt.show()
else:
    print('Provide a sample test image and ensure models are trained.')


In [ ]:

# Final inference helper: predict single image or folder, show Grad-CAM for chosen model
import cv2
def predict_image(path, model_name='resnet', show_gradcam=True):
    mfile = f'models/{model_name}_finetuned.h5'
    if not os.path.exists(mfile):
        print('Model not found:', mfile); return
    model = load_model(mfile)
    img = cv2.imread(path); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB); img_r = cv2.resize(img, IMG_SIZE)
    x = np.expand_dims(img_r,0).astype('float32')
    if model_name=='resnet': x_p = resnet50.preprocess_input(x.copy())
    elif model_name=='eff': x_p = efficientnet.preprocess_input(x.copy())
    else: x_p = mobilenet_v3.preprocess_input(x.copy())
    pred = model.predict(x_p)[0]
    cls = np.argmax(pred)
    print('Predicted class:', cls, 'prob:', np.max(pred))
    if show_gradcam:
        # reuse gradcam_layers for last conv names guess
        try:
            last_conv = [l.name for l in model.layers if l.name.startswith('conv')][-1]
            from matplotlib import pyplot as plt
            heat = gradcam_layers(model, x_p, [last_conv])
            heat = cv2.resize(heat, (img_r.shape[1], img_r.shape[0]))
            heat_img = np.uint8(255*heat); heat_img = cv2.applyColorMap(heat_img, cv2.COLORMAP_JET)
            overlay = cv2.addWeighted(img_r, 0.6, heat_img, 0.4, 0)
            plt.figure(figsize=(8,4)); plt.subplot(1,2,1); plt.imshow(img_r); plt.axis('off'); plt.title('Image')
            plt.subplot(1,2,2); plt.imshow(overlay); plt.axis('off'); plt.title('Grad-CAM'); plt.show()
        except Exception as e:
            print('Grad-CAM error:', e)

# Example usage:
# predict_image('path/to/image.png', model_name='resnet', show_gradcam=True)


## End of notebook
Files saved to `models/`. Run cells sequentially; adjust paths and batch sizes as needed.